# SwitchLingua: training the transformer models on Colab

This notebook trains the two prefix-only switch predictors on a Colab GPU and saves everything to Google Drive:

| Model | What it is | Expected time on a T4 |
|---|---|---|
| XLM-R base + causal mask | bidirectional encoder forced to read left to right (the proposal's model) | about 1 to 2 hours |
| XGLM-564M | multilingual model that reads left to right natively | about 3 to 5 hours |

**Before you start**
1. *Runtime > Change runtime type > T4 GPU* (or better).
2. Add your Hugging Face token as a Colab secret: click the key icon in the left sidebar, add a secret named `HF_TOKEN`, and switch on notebook access. Use a **new** token, and make sure your account has accepted the dataset terms on https://huggingface.co/datasets/Shelton1013/SwitchLingua_text. Never paste the token into a cell.
3. The code must be on GitHub (set `REPO_URL` below) or uploaded to Drive as `switchlingua.zip` in `MyDrive/switchlingua/`.
4. You need about 12 GB free on Google Drive for the checkpoints (see step 7).

**If Colab disconnects:** run the setup cells (1 to 5) again, set `RESUME = True` in the training cell you were running, and run it again. Training continues from the last checkpoint saved on Drive.

Run the cells from top to bottom. Each cell says what it does.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Connect Google Drive (results and checkpoints are saved there)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/switchlingua")
for sub in ["checkpoints", "runs", "processed"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print("Drive folder:", DRIVE)

## 3. Get the code

Option A (default): clone from GitHub. For a **private** repo, add a GitHub token with read access as a Colab secret named `GH_TOKEN`.
Option B: if `REPO_URL` is empty, the notebook unzips `MyDrive/switchlingua/switchlingua.zip`.

In [ ]:
import base64, os, subprocess
REPO_URL = "https://github.com/msadou02-hue/switchlingua.git"   # <- edit, or set to "" for option B
CODE = Path("/content/switchlingua")

def git(*args):
    # Run git with the GH_TOKEN secret passed for this command only (never stored or printed).
    extra = []
    try:
        from google.colab import userdata
        auth = base64.b64encode(f"x-access-token:{userdata.get('GH_TOKEN')}".encode()).decode()
        extra = ["-c", f"http.extraHeader=Authorization: Basic {auth}"]
    except Exception:
        pass  # no secret: fine for a public repo
    r = subprocess.run(["git", *extra, *args], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    return r.returncode

if CODE.exists():
    print("code already present:", CODE)
    if (CODE / ".git").exists():
        git("-C", str(CODE), "pull", "--ff-only", "origin", "master")
elif REPO_URL and "<your-account>" not in REPO_URL:
    assert git("clone", "--quiet", REPO_URL, str(CODE)) == 0, "clone failed: check REPO_URL and the GH_TOKEN secret"
else:
    zip_path = DRIVE / "switchlingua.zip"
    assert zip_path.exists(), f"Set REPO_URL or upload the code to {zip_path}"
    !unzip -q {zip_path} -d {CODE}

%cd {CODE}
!git log --oneline -1 2>/dev/null || echo "(no git history, using zip)"

## 4. Install packages

Colab already has a CUDA build of PyTorch, so it is **not** reinstalled (the pinned CPU version in `requirements.txt` is for the Windows laptop).

In [ ]:
!pip install -q "transformers==5.18.0" "datasets==5.0.1" "wordfreq==3.1.1" pytest
import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)

## 5. Hugging Face login and downloads

The scripts default to offline mode (right for the laptop). Here we switch it off and download the dataset (1.4 GB) and both models once per session.

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")      # read from the Colab secret, never printed
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["HF_DATASETS_OFFLINE"] = "0"

from datasets import load_dataset
ds = load_dataset("Shelton1013/SwitchLingua_text", split="train")
print("dataset rows:", len(ds))   # expected 234172

from transformers import AutoModel, AutoTokenizer
for name in ["FacebookAI/xlm-roberta-base", "facebook/xglm-564M"]:
    AutoTokenizer.from_pretrained(name)
    AutoModel.from_pretrained(name)
    print("downloaded", name)
del ds

## 6. Tests and processed data

The tests check labels, splits, alignment and, most importantly, that no information from word t+1 leaks into the prediction at word t.

The processed data is rebuilt from the dataset (about 10 minutes) the first time and cached on Drive afterwards. The **splits are not regenerated**: `prepare_data` reuses `results/splits/row_splits.csv.gz` from the repo, so Colab uses exactly the same train/val/test rows as the laptop.

In [ ]:
!python -m pytest -q tests 2>&1 | tail -3

In [ ]:
import shutil
local = CODE / "data" / "processed" / "sentences"
cached = DRIVE / "processed" / "sentences"
if cached.exists() and any(cached.glob("part-*.parquet")):
    shutil.copytree(cached, local, dirs_exist_ok=True)
    print("copied processed data from Drive")
else:
    !python -u -m scripts.prepare_data
    shutil.copytree(local, cached, dirs_exist_ok=True)
    print("built processed data and cached it on Drive")

!git status --short results/splits   # must print nothing: the splits are unchanged

## 7. Save checkpoints to Drive

`checkpoints/` inside the repo becomes a link to Drive, so the resume file (`last.pt`) survives a disconnect.
Sizes: XLM-R about 3.3 GB and XGLM about 6.8 GB for `last.pt` (weights plus optimizer state), plus the final weights.

In [ ]:
ckpt_link = CODE / "checkpoints"
if ckpt_link.is_symlink() or not ckpt_link.exists():
    ckpt_link.unlink(missing_ok=True)
    ckpt_link.symlink_to(DRIVE / "checkpoints", target_is_directory=True)
else:
    raise SystemExit(f"{ckpt_link} is a real folder; move its contents to {DRIVE/'checkpoints'} first")
print(ckpt_link, "->", ckpt_link.resolve())

def save_run(run_name):
    # Copy a finished run's metrics to Drive (results/runs is small; checkpoints are already there).
    src = CODE / "results" / "runs" / run_name
    if not src.exists():
        print(f"NOT SAVED: {src} does not exist. Training stopped before the end; read the error above.")
        return
    shutil.copytree(src, DRIVE / "runs" / run_name, dirs_exist_ok=True)
    print("saved", src, "->", DRIVE / "runs" / run_name)

## 8. Short check run (about 5 minutes)

200 steps of the real XLM-R on a small sample. It confirms the GPU pipeline works and measures the speed. The numbers are **not** results: the model has barely trained. To rerun this cell, delete `results/runs/colab_check_xlmr` first (runs are never overwritten).

In [ ]:
!python -u -m scripts.train --model FacebookAI/xlm-roberta-base --max-steps 200 --log-every 50 \
    --batch-size 32 --fp16 --num-workers 2 --eval-per-strata 100 --threshold-per-strata 100 \
    --run-name colab_check_xlmr

In [ ]:
import pandas as pd
log = pd.read_csv(CODE / "results/runs/colab_check_xlmr/train_log.csv")
sec_per_step = (log.elapsed_s.iloc[-1] - log.elapsed_s.iloc[0]) / (log.step.iloc[-1] - log.step.iloc[0])
print(f"{sec_per_step:.2f} s/step -> 20000 steps take about {20000 * sec_per_step / 3600:.1f} h for XLM-R")
print("XGLM-564M is roughly 2 to 3 times slower per step.")

## 9. Full XLM-R run

Settings: 20,000 steps, batch 32, temperature sampling alpha = 0.3, learning rate 2e-5 (heads 1e-3), mixed precision.
A checkpoint is saved to Drive every 1,000 steps. After a disconnect: rerun cells 1 to 7, set `RESUME = True` here, run this cell again.

In [ ]:
RESUME = False                        # set to True after a disconnect
RESUME_FLAG = "--resume" if RESUME else ""
RUN = "xlmr_a0.3_s20000_seed13"
!python -u -m scripts.train --model FacebookAI/xlm-roberta-base --run-name {RUN} \
    --max-steps 20000 --batch-size 32 --fp16 --num-workers 2 --save-every 1000 --log-every 200 \
    {RESUME_FLAG}
save_run(RUN)

## 10. Full XGLM-564M run

Same settings, but batch 16 with 2 accumulation steps (same effective batch of 32) so the larger model fits in T4 memory.

In [ ]:
RESUME = False                        # set to True after a disconnect
RESUME_FLAG = "--resume" if RESUME else ""
RUN = "xglm_a0.3_s20000_seed13"
!python -u -m scripts.train --model facebook/xglm-564M --run-name {RUN} \
    --max-steps 20000 --batch-size 16 --grad-accum 2 --fp16 --num-workers 2 --save-every 1000 --log-every 200 \
    {RESUME_FLAG}
save_run(RUN)

## 11. Compare with the baselines

Overall = macro average over the 16 strata (15 pairs, Hindi split by script). Lower sigma = more universal.
All runs use the validation split; the test split stays untouched until the final evaluation.

In [ ]:
import json
runs = ["majority_val_seed13", "logreg_val_seed13", "xlmr_a0.3_s20000_seed13", "xglm_a0.3_s20000_seed13"]
cols = ["macro_sw_f1", "macro_sw_pr_auc", "macro_sw_precision", "macro_sw_recall", "sigma_universality_sw_f1",
        "macro_dur_macro_f1", "macro_dur_accuracy"]
rows, per = {}, {}
for r in runs:
    d = CODE / "results" / "runs" / r
    if not d.exists():
        print("missing (not run yet):", r)
        continue
    rows[r] = json.loads((d / "overall.json").read_text())
    per[r] = pd.read_csv(d / "per_strata.csv", index_col=0).sw_f1
display(pd.DataFrame(rows).T[cols].round(3))
display(pd.DataFrame(per).round(3).rename_axis("switch F1 per stratum"))

## 12. Bring the results back to the repo

The run folders are small (CSV and JSON) and belong in `results/runs/` in the repo, so the team can see them.
Either download them from `MyDrive/switchlingua/runs/` and commit them on the laptop, or push from here (needs a GitHub token with write access).
After a run finishes successfully you can delete its `last.pt` on Drive to free space (`model.pt` holds the final weights).

In [ ]:
!ls -la {DRIVE}/runs {DRIVE}/checkpoints/*